# Passport POC — start from (simulated) CDW

**All data is synthetic and every CDW table and column name is SIMULATED.** Names are illustrative; the mapping to the
extract template's `<<placeholders>>` is in `sql/cdw_placeholder_map.csv`, where real CDW Database Library names can be
filled in later inside an IRS/BAH-controlled environment.

Run all on Serverless. The notebook:

1. **Builds a simulated CDW** (`cdw_schema`, incl. a federal business-day calendar) by un-flattening the validated synthetic `pp_*` tables into source-style
   tables (IMF entity/module/CSED/transactions/notices, ALS liens, ACDS/ACMS appeals) with **masked TINs** — the same
   mask in every source and stable across rebuilds — plus noise the extract must filter out
2. **Runs the extract** (`sql/cdw_extract_passport.sql`, the filled template) into `target_schema`
3. **Round-trip check**: the extract must rebuild the original tables exactly for every taxpayer in the driver population
   (stops on failure), and lists the taxpayers the driver leaves out
4. **Runs the rules and validates** (answer key 0 mismatches, `v_threshold_gaps` 0 rows, finding counts vs. reference)

Prerequisite: the validated synthetic tables **v0.5.3** (IRM-based lien deadlines) loaded in `source_schema` (e.g. by `setup_passport_poc`).

**Confirm in CDW** (simulated here, flagged in `cdw_placeholder_map.csv`): the Letter 3172 must-file date, the NFTL mailing
date, and a business-day/holiday calendar.

In [ ]:
dbutils.widgets.text("catalog", "workspace", "1. Catalog")
dbutils.widgets.text("source_schema", "passport_poc", "2. Source schema (synthetic pp_* tables)")
dbutils.widgets.text("cdw_schema", "cdw_sim", "3. Simulated CDW schema")
dbutils.widgets.text("target_schema", "passport_poc_cdw", "4. Extract target schema")
dbutils.widgets.text("extract_sql", "./sql/cdw_extract_passport.sql", "5. Extract SQL")
dbutils.widgets.text("rules_sql", "./sql/passport_rules.sql", "6. Rules SQL")

import os, re
C = dbutils.widgets.get("catalog").strip()
names = {k: dbutils.widgets.get(k).strip() for k in ("source_schema", "cdw_schema", "target_schema")}
for k, v in [("catalog", C), *names.items()]:
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", v):
        raise ValueError(f"{k} '{v}' must be letters, digits and underscores only")
if len(set(names.values())) < 3:
    raise ValueError("source, CDW and target schemas must all be different")
SRC = f"`{C}`.`{names['source_schema']}`"
CDW = f"`{C}`.`{names['cdw_schema']}`"
OUT = f"`{C}`.`{names['target_schema']}`"
def path(w):
    v = dbutils.widgets.get(w).strip(); return v if v.startswith("/") else os.path.abspath(v)
EXTRACT_SQL, RULES_SQL = path("extract_sql"), path("rules_sql")
print("source:", SRC, "| CDW:", CDW, "| target:", OUT)

## 1. Build the simulated CDW

In [ ]:
# ---- Business-day calendar (federal holidays, observed) ---------------------------
# SIMULATED reference table. CONFIRM IN CDW: does CDW (or IRS reference data) carry an
# official business-day / holiday calendar the extract can join to?
import datetime as dt

def _nth_weekday(y, mth, wd, n):           # n-th weekday (0=Mon) of month; n=-1 -> last
    if n > 0:
        d0 = dt.date(y, mth, 1); off = (wd - d0.weekday()) % 7
        return d0 + dt.timedelta(days=off + 7 * (n - 1))
    d0 = (dt.date(y + (mth == 12), mth % 12 + 1, 1) - dt.timedelta(days=1))
    return d0 - dt.timedelta(days=(d0.weekday() - wd) % 7)

def federal_holidays(y):
    def obs(d):                            # Saturday -> Friday, Sunday -> Monday
        return d - dt.timedelta(days=1) if d.weekday() == 5 else d + dt.timedelta(days=1) if d.weekday() == 6 else d
    h = [obs(dt.date(y, 1, 1)), _nth_weekday(y, 1, 0, 3), _nth_weekday(y, 2, 0, 3), _nth_weekday(y, 5, 0, -1),
         obs(dt.date(y, 7, 4)), _nth_weekday(y, 9, 0, 1), _nth_weekday(y, 10, 0, 2), obs(dt.date(y, 11, 11)),
         _nth_weekday(y, 11, 3, 4), obs(dt.date(y, 12, 25))]
    if y >= 2021: h.append(obs(dt.date(y, 6, 19)))
    return set(h)

def business_calendar(start=dt.date(2015, 1, 1), end=dt.date(2036, 12, 31)):
    hol = set().union(*(federal_holidays(y) for y in range(start.year - 1, end.year + 2)))
    rows, seq, d = [], 0, start
    while d <= end:
        bus = d.weekday() < 5 and d not in hol
        seq += bus
        rows.append((d, bus, seq, d in hol))
        d += dt.timedelta(days=1)
    return rows   # (CAL_DT, IS_BUS_DAY, BUS_DAY_SEQ = business days through this date, IS_FED_HOLIDAY)

def add_business_days(cal_index, d, n):
    # n-th business day after d (same rule as the extract SQL)
    seq = cal_index["seq"][d]
    return cal_index["by_seq"][seq + n]

def index_calendar(rows):
    return {"seq": {r[0]: r[2] for r in rows}, "by_seq": {r[2]: r[0] for r in rows if r[1]}}

def must_file_date(cal_index, mail_dt):
    # IRM 5.19.4.6.2 (12-30-2024): estimated filing date = mailing + 3 business days;
    # must-file date = estimated filing date + 5 business days + 30 calendar days.
    # Weekend/holiday handling of the final day: NOT specified there - open item.
    est = add_business_days(cal_index, mail_dt, 3)
    return est, add_business_days(cal_index, est, 5) + dt.timedelta(days=30)

def mailing_from_filed(cal_index, filed):
    # Synthetic only: choose the mailing date so the IRS estimated filing date equals the
    # recorded filing date (3 business days back from the filing date's business-day position).
    return cal_index["by_seq"][cal_index["seq"][filed] - 3]


import hashlib, random, datetime as dt
from decimal import Decimal

SALT = "PASSPORT-POC-SIM"   # fixed so masked TINs are stable across rebuilds (as CDW's are)

def mask(token):
    # Simulated CDW masked TIN: 'M' + 8 digits, deterministic. Not reversible to any real TIN.
    return "M" + str(int(hashlib.sha256((SALT + token).encode()).hexdigest(), 16) % 10**8).zfill(8)

def s(v):
    if v is None: return None
    v = str(v).strip()
    return None if v in ("", "None", "nan", "NaT") else v
def i(v):
    v = s(v); return None if v is None else int(float(v))
def d(v):
    v = s(v); return None if v is None else dt.date.fromisoformat(v[:10])
def m(v):
    v = s(v); return None if v is None else Decimal(v).quantize(Decimal("0.01"))
def p(v):  # tax period as YYYYMM string
    v = s(v); return None if v is None else str(int(float(v)))

def build_cdw_sim(src, seed=7345):
    """src: dict of table name -> list of dict rows (the pp_* tables). Returns name -> (ddl, rows)."""
    rnd = random.Random(seed)
    toks = sorted({r["tin_token"] for r in src["pp_entity"]})
    tmap = {t: mask(t) for t in toks}
    assert len(set(tmap.values())) == len(tmap), "mask collision"
    M = lambda t: None if s(t) is None else tmap[s(t)]

    ent = [(M(r["tin_token"]), s(r["tin_type"]), s(r["name_control"]), d(r["date_of_death"]),
            s(r["foreign_address_ind"]), s(r["address_state"])) for r in src["pp_entity"]]
    mod = [(M(r["tin_token"]), i(r["mft"]), p(r["tax_period"]), s(r["joint_ind"]),
            M(r["secondary_tin_token"]), m(r["assessed_balance"]), s(r["status_code"])) for r in src["pp_module"]]
    csed = [(M(r["tin_token"]), i(r["mft"]), p(r["tax_period"]), i(r["assessment_tc"]),
             d(r["assessment_date"]), d(r["csed_date"])) for r in src["pp_csed"]]
    txn = [(M(r["tin_token"]), i(r["mft"]), p(r["tax_period"]), i(r["tc"]), i(r["action_code"]),
            i(r["closing_code"]), d(r["txn_date"]), s(r["posting_cycle"]), m(r["amount"]),
            s(r["dln"]), s(r["misc_field"])) for r in src["pp_transaction"]]
    ntc = [(M(r["tin_token"]), i(r["mft"]), p(r["tax_period"]), s(r["notice_code"]),
            d(r["issue_date"]), s(r["undeliverable_ind"])) for r in src["pp_notice"]]
    CAL = business_calendar(); CIX = index_calendar(CAL)
    # ALS lien: NFTL mailing date and the Letter 3172 must-file date are NEW elements -
    # CONFIRM BOTH EXIST IN CDW. Every 6th lien leaves the must-file date blank so the
    # extract's computed fallback (IRM 5.19.4.6.2) is exercised.
    lien = []
    for n, r in enumerate(sorted(src["pp_lien"], key=lambda r: s(r["lien_id"]))):
        filed = d(r["nftl_filed_date"])
        lien.append((s(r["lien_id"]), M(r["tin_token"]), mailing_from_filed(CIX, filed), filed,
                     d(r["cdp_notice_date"]), None if n % 6 == 5 else d(r["cdp_request_deadline"]),
                     d(r["lien_release_date"])))
    lmod = [(s(r["lien_id"]), i(r["mft"]), p(r["tax_period"])) for r in src["pp_lien"]]
    hcode_acds = {"LIEN_CDP": "L", "LEVY_CDP": "V", "EQUIV": "E"}
    hcode_acms = {"LIEN_CDP": "LIEN", "LEVY_CDP": "LEVY", "EQUIV": "EQUIV"}
    acds, acms = [], []
    for r in src["pp_appeals_cdp"]:
        row = (s(r["appeal_key"]), M(r["tin_token"]), i(r["mft"]), p(r["tax_period"]), "CDP",
               None, d(r["cdp_request_date"]), s(r["timely_ind"]), d(r["cdp_closed_date"]))
        if s(r["source_system"]) == "ACDS":
            acds.append(row[:5] + (hcode_acds[s(r["hearing_type"])],) + row[6:])
        else:
            acms.append(row[:5] + (hcode_acms[s(r["hearing_type"])],) + row[6:])

    # ---- Noise the extract must filter out -----------------------------------------
    dln_n = [0]
    def dln():
        dln_n[0] += 1; return f"SIMN{dln_n[0]:010d}"
    states = ["WA", "OR", "CA", "TX", "NY", "FL", "OH", "GA", "AZ"]
    letters = "ABCDEFGHJKLMNPRSTW"
    # (1) 40 taxpayers outside the population: low balance or paid, never certified
    for k in range(1, 41):
        tok = f"N{k:04d}"; mt = mask(tok); tmap[tok] = mt
        ent.append((mt, "SSN", "".join(rnd.choice(letters) for _ in range(4)), None, "N", rnd.choice(states)))
        tp = rnd.choice(["202012", "202112", "202212"])
        yr = int(tp[:4]) + 1
        assess = Decimal(rnd.randrange(8000, 45000)).quantize(Decimal("0.01"))
        paid = assess if k <= 10 else Decimal(rnd.randrange(0, int(assess) // 2)).quantize(Decimal("0.01"))
        bal = assess - paid
        mod.append((mt, 30, tp, "N", None, bal, "12" if bal == 0 else "22"))
        ad = dt.date(yr, 5, rnd.randrange(1, 28))
        csed.append((mt, 30, tp, 150, ad, ad.replace(year=ad.year + 10)))
        cyc = f"{ad.year}{ad.isocalendar()[1]:02d}"
        txn.append((mt, 30, tp, 150, None, None, ad, cyc, assess, dln(), None))
        if paid:
            pdte = ad + dt.timedelta(days=rnd.randrange(60, 700))
            txn.append((mt, 30, tp, 670, None, None, pdte, f"{pdte.year}{pdte.isocalendar()[1]:02d}", -paid, dln(), None))
        ntc.append((mt, 30, tp, "CP14", ad + dt.timedelta(days=14), "N"))
        if k > 34:  # (2) liens on a few out-of-population taxpayers
            lid = f"LN{k:05d}"; f = ad + dt.timedelta(days=400)
            mail = mailing_from_filed(CIX, f)
            lien.append((lid, mt, mail, f, f + dt.timedelta(days=4), must_file_date(CIX, mail)[1], None)); lmod.append((lid, 30, tp))
    # (3) Non-extract notice codes for population taxpayers
    real = {tmap[t] for t in toks}
    pop = sorted({r[0] for r in mod if r[0] in real})
    for mt in rnd.sample(pop, 60):
        mrow = next(r for r in mod if r[0] == mt)
        ntc.append((mt, mrow[1], mrow[2], rnd.choice(["CP14", "CP501", "CP503", "CP504"]),
                    dt.date(2024, rnd.randrange(1, 13), rnd.randrange(1, 28)), "N"))
    # (4) Non-CDP Appeals cases (Collection Appeals Program, OIC) for population taxpayers
    for j, mt in enumerate(rnd.sample(pop, 15)):
        mrow = next(r for r in mod if r[0] == mt)
        rq = dt.date(2025, rnd.randrange(1, 13), rnd.randrange(1, 28))
        if j < 10:
            acms.append((f"ACMS-N{j:05d}", mt, mrow[1], mrow[2], "CAP", None, rq, None, rq + dt.timedelta(days=45)))
        else:
            acds.append((f"ACDS-N{j:05d}", mt, mrow[1], mrow[2], "OIC", None, rq, None, None))

    T = {
     "IMF_ENTITY": ("MASKED_TIN STRING, TIN_TYPE_CD STRING, NM_CTRL STRING, DOD_DT DATE, FRGN_ADDR_IND STRING, ADDR_ST_CD STRING", ent),
     "IMF_MODULE": ("MASKED_TIN STRING, MFT_CD INT, TAX_PRD STRING, JNT_FLR_IND STRING, SEC_MASKED_TIN STRING, MOD_BAL_AMT DECIMAL(15,2), MOD_STAT_CD STRING", mod),
     "IMF_CSED":   ("MASKED_TIN STRING, MFT_CD INT, TAX_PRD STRING, ASSMT_TC INT, ASSMT_DT DATE, CSED_DT DATE", csed),
     "IMF_TXN":    ("MASKED_TIN STRING, MFT_CD INT, TAX_PRD STRING, TC INT, ACTN_CD INT, CLSNG_CD INT, TRANS_DT DATE, PSTNG_CYC STRING, TRANS_AMT DECIMAL(15,2), DLN STRING, MISC_FLD STRING", txn),
     "IMF_NOTICE": ("MASKED_TIN STRING, MFT_CD INT, TAX_PRD STRING, NTC_CD STRING, NTC_ISS_DT DATE, UNDLV_IND STRING", ntc),
     "ALS_LIEN":   ("LIEN_ID STRING, MASKED_TIN STRING, NFTL_MAIL_DT DATE, NFTL_FILED_DT DATE, CDP_NTC_DT DATE, L3172_MUST_FILE_DT DATE, LIEN_RLSE_DT DATE", lien),
     "ALS_LIEN_MODULE": ("LIEN_ID STRING, MFT_CD INT, TAX_PRD STRING", lmod),
     "ACDS_CASE":  ("WUNO STRING, MASKED_TIN STRING, MFT_CD INT, TAX_PRD STRING, CASE_CAT_CD STRING, HRG_TYPE_CD STRING, RQST_RCVD_DT DATE, TIMELY_IND STRING, CLOSED_DT DATE", acds),
     "REF_BUS_CALENDAR": ("CAL_DT DATE, IS_BUS_DAY BOOLEAN, BUS_DAY_SEQ INT, IS_FED_HOLIDAY BOOLEAN", CAL),
     "ACMS_APPEAL":("APPEAL_NUM STRING, MASKED_TIN STRING, MFT_CD INT, TAX_PRD STRING, CASE_TYPE_CD STRING, HRG_TYPE_CD STRING, RQST_RCVD_DT DATE, TIMELY_IND STRING, CLOSED_DT DATE", acms),
    }
    return T, tmap


In [ ]:
src = {}
for t in ["pp_entity","pp_module","pp_csed","pp_transaction","pp_notice","pp_lien","pp_appeals_cdp"]:
    src[t] = [r.asDict() for r in spark.table(f"{SRC}.{t}").collect()]
    print(f"read {t:<16} {len(src[t]):>6}")

TABLES, TOKEN_MAP = build_cdw_sim(src)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CDW} COMMENT 'SIMULATED CDW - synthetic data, illustrative names'")
for name, (ddl, rows) in TABLES.items():
    (spark.createDataFrame(rows, ddl).write.mode("overwrite").option("overwriteSchema", "true")
          .saveAsTable(f"{CDW}.`{name}`"))
    spark.sql(f"COMMENT ON TABLE {CDW}.`{name}` IS 'SIMULATED CDW table (synthetic, illustrative name)'")
    print(f"built {name:<16} {len(rows):>6} rows")

## 2. Run the extract

In [ ]:
rp = spark.table(f"{SRC}.pp_run_params").first()
AS_OF = str(rp["as_of_date"])[:10]
THRESHOLD = spark.sql(f"SELECT threshold_amount FROM {SRC}.ref_threshold WHERE CAST(threshold_year AS INT) = {int(rp['threshold_year'])}").first()[0]
params = {"{{cdw}}": CDW, "{{out}}": OUT, "{{AS_OF}}": AS_OF, "{{LOOKBACK_START}}": "2018-01-01", "{{THRESHOLD}}": str(int(THRESHOLD))}
print(params)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {OUT}")
text = open(EXTRACT_SQL).read()
text = re.sub(r"/\*.*?\*/", "", text, flags=re.S)
text = "\n".join(l for l in text.splitlines() if not l.strip().startswith("--"))
for k, v in params.items():
    text = text.replace(k, v)
assert "{{" not in text, "unfilled placeholder in extract SQL"
for n, st in enumerate([x.strip() for x in re.split(r";\s*(?:\n|$)", text) if x.strip()], 1):
    spark.sql(st)
    print(f"ok {n:>2}  {' '.join(st.split()[:6])}")

# reference tables and run parameters are not CDW data: copy them from the source schema
for t in ["pp_run_params","ref_indicator","ref_indicator_reversal","ref_noncert_mft","ref_threshold"]:
    spark.sql(f"CREATE OR REPLACE TABLE {OUT}.{t} AS SELECT * FROM {SRC}.{t}")
display(spark.table(f"{CDW}.WRK_PP_CONTROL"))
ctl = {r["check_name"]: r["n"] for r in spark.table(f"{CDW}.WRK_PP_CONTROL").collect()}
print(f"Letter 3172 must-file date: {ctl['must_file_pulled']} pulled, {ctl['must_file_computed_fallback']} computed (IRM 5.19.4.6.2); "
      f"{ctl['must_file_on_weekend_or_holiday']} fall on a weekend/holiday (open item)")
if ctl["must_file_missing"]:
    raise RuntimeError("Some liens have no must-file date (no pulled date and no mailing date/calendar match)")
if ctl["must_file_pulled_ne_computed"]:
    raise RuntimeError("Pulled must-file dates disagree with the IRM 5.19.4.6.2 calculation - "
                       "the source schema probably holds pre-v0.5.3 lien data; reload data v0.5.3 and rerun")

## 3. Round-trip check (stops on failure)

In [ ]:
spark.createDataFrame(list(TOKEN_MAP.items()), "tin_token STRING, masked_tin STRING").createOrReplaceTempView("sim_token_map")

SPEC = {  # column -> type used to compare original vs. rebuilt; 'TOK' = token mapped through the mask
 "pp_entity": {"tin_token":"TOK","tin_type":"STRING","name_control":"STRING","date_of_death":"DATE","foreign_address_ind":"STRING","address_state":"STRING"},
 "pp_module": {"tin_token":"TOK","mft":"INT","tax_period":"STRING","joint_ind":"STRING","secondary_tin_token":"TOK","assessed_balance":"DECIMAL(15,2)","status_code":"STRING","extract_date":"DATE"},
 "pp_csed": {"tin_token":"TOK","mft":"INT","tax_period":"STRING","assessment_tc":"INT","assessment_date":"DATE","csed_date":"DATE"},
 "pp_transaction": {"tin_token":"TOK","mft":"INT","tax_period":"STRING","tc":"INT","action_code":"INT","closing_code":"INT","txn_date":"DATE","posting_cycle":"STRING","amount":"DECIMAL(15,2)","dln":"STRING","misc_field":"STRING"},
 "pp_notice": {"tin_token":"TOK","mft":"INT","tax_period":"STRING","notice_code":"STRING","issue_date":"DATE","undeliverable_ind":"STRING"},
 "pp_lien": {"lien_id":"STRING","tin_token":"TOK","mft":"INT","tax_period":"STRING","nftl_filed_date":"DATE","cdp_notice_date":"DATE","cdp_request_deadline":"DATE","lien_release_date":"DATE"},
 "pp_appeals_cdp": {"appeal_key":"STRING","source_system":"STRING","tin_token":"TOK","mft":"INT","tax_period":"STRING","hearing_type":"STRING","cdp_request_date":"DATE","timely_ind":"STRING","cdp_closed_date":"DATE"},
}
def norm(c, typ):
    return f"NULLIF(CAST({c} AS STRING), '')" if typ == "STRING" else f"CAST(NULLIF(CAST({c} AS STRING), '') AS {typ})"

results = []
for t, cols in SPEC.items():
    o_sel, joins = [], []
    for k, (c, typ) in enumerate(cols.items()):
        if typ == "TOK":
            joins.append(f"LEFT JOIN sim_token_map m{k} ON m{k}.tin_token = NULLIF(CAST(o.{c} AS STRING), '')")
            o_sel.append(f"m{k}.masked_tin AS {c}")
        else:
            o_sel.append(f"{norm('o.' + c, typ)} AS {c}")
    n_sel = [f"NULLIF(CAST(n.{c} AS STRING), '') AS {c}" if typ == "TOK" else f"{norm('n.' + c, typ)} AS {c}" for c, typ in cols.items()]
    orig = (f"SELECT {', '.join(o_sel)} FROM {SRC}.{t} o {' '.join(joins)} "
            f"WHERE m{list(cols.values()).index('TOK')}.masked_tin IN (SELECT tin FROM {CDW}.WRK_PP_DRV)")
    new = f"SELECT {', '.join(n_sel)} FROM {OUT}.{t} n"
    missing = spark.sql(f"{orig} EXCEPT ALL {new}").count()
    extra = spark.sql(f"{new} EXCEPT ALL {orig}").count()
    results.append((t, spark.sql(new).count(), missing, extra))
display(spark.createDataFrame(results, "table string, rebuilt_rows long, missing_vs_original long, extra_vs_original long"))

left_out = spark.sql(f"""
  SELECT k.scenario_id, k.scenario, k.expected_findings, COUNT(*) AS taxpayers
  FROM {SRC}.pp_answer_key k JOIN sim_token_map m ON m.tin_token = k.tin_token
  WHERE m.masked_tin NOT IN (SELECT tin FROM {CDW}.WRK_PP_DRV)
  GROUP BY ALL ORDER BY 1""")
print("Taxpayers the driver population leaves out (expected: only scenarios with no findings):")
display(left_out)

bad = [r for r in results if r[2] or r[3]]
lost = left_out.where("expected_findings <> 'NONE'").count()
if bad or lost:
    raise RuntimeError(f"ROUND TRIP FAILED: {len(bad)} table(s) differ; {lost} left-out scenario(s) expected findings")
print("ROUND TRIP PASSED: the extract rebuilt every driver taxpayer's rows exactly")

## 4. Rules, answer key and validation

In [ ]:
# answer key: synthetic test artifact, re-keyed to masked TINs (not part of CDW or the extract)
spark.sql(f"""CREATE OR REPLACE TABLE {OUT}.pp_answer_key AS
  SELECT m.masked_tin AS tin_token, k.scenario_id, k.scenario, k.expected_findings, k.hand_built
  FROM {SRC}.pp_answer_key k JOIN sim_token_map m ON m.tin_token = k.tin_token
  WHERE m.masked_tin IN (SELECT tin_token FROM {OUT}.pp_entity)""")

spark.sql(f"USE CATALOG `{C}`"); spark.sql(f"USE SCHEMA `{names['target_schema']}`")
rules = open(RULES_SQL).read()
keep = [l for l in rules.splitlines() if not l.strip().startswith("--") and not re.match(r"\s*USE\s+(CATALOG|SCHEMA)\b", l, re.I)]
for st in [x.strip() for x in re.split(r";\s*(?:\n|$)", "\n".join(keep)) if x.strip()]:
    spark.sql(st)

gaps = spark.sql(f"SELECT * FROM {OUT}.v_threshold_gaps").count()
mm = spark.sql(f"""
  WITH exp AS (SELECT DISTINCT k.tin_token, trim(f) AS finding FROM {OUT}.pp_answer_key k
               LATERAL VIEW explode(split(coalesce(k.expected_findings, ''), '[;,|]')) x AS f
               WHERE trim(f) NOT IN ('', 'NONE', 'None', 'none')),
       act AS (SELECT DISTINCT tin_token, finding FROM {OUT}.v_findings)
  SELECT 'MISSED' AS issue, * FROM exp LEFT ANTI JOIN act USING (tin_token, finding)
  UNION ALL SELECT 'EXTRA', * FROM act LEFT ANTI JOIN exp USING (tin_token, finding)""")
mismatches = mm.count()
print(f"v_threshold_gaps rows: {gaps} | answer-key mismatches: {mismatches}")
if gaps or mismatches:
    display(mm.limit(50)); raise RuntimeError("VALIDATION FAILED")

reference = {"CERTIFIED_BELOW_THRESHOLD_AT_CERT": 9, "CERTIFIED_MODULE_NO_PREDICATE": 9, "DECERT_DUE_OVERDUE": 36,
             "DECERT_DUE_PENDING": 18, "INFO_BELOW_THRESHOLD_STILL_CERTIFIED": 9, "INFO_EXPEDITE_INELIGIBLE": 9,
             "MISSED_CERTIFICATION": 52, "MODULE_REVERSAL_DUE_OVERDUE": 9, "NEW_ELIGIBLE_MODULE_NOT_CERTIFIED": 9,
             "REVERSED_LATE": 9, "REVERSED_TIMELY": 9}
got = {r["finding"]: r["n"] for r in spark.sql(f"SELECT finding, COUNT(*) n FROM {OUT}.v_findings GROUP BY finding").collect()}
rows = [(k, got.get(k, 0), reference.get(k, 0), "same" if got.get(k, 0) == reference.get(k, 0) else "DIFFERENT")
        for k in sorted(set(got) | set(reference))]
display(spark.createDataFrame(rows, "finding string, from_cdw long, reference long, status string"))
print("VALIDATION PASSED - CDW -> extract -> rules reproduces the reference findings"
      if all(r[3] == "same" for r in rows) else "Finding counts differ from the reference - investigate")